# RATISS-ONE — entraînement Colab (tokenizer v1)
Honnêteté d'abord : ici, *entraîner* ne veut PAS dire descente de gradient.
Notre entraînement = **éducation du tissu** (relations, pas de poids en Go)
+ **vérification des règles** (chaque paire état→phrase reproduite à 100 %).
Le GPU ne sert pas (entiers purs) : Colab sert l'ÉCHELLE et la VITESSE.
Critères de sortie : batterie 107/107, paires 100 % reproduites, 1 wav audible.
Vérifié le 7 oct. 2026 sur clone frais : 107/107 + preuve audio 7,1 s.

In [ ]:
!git clone https://github.com/jonathansearch/RATISS-ONE.git
%cd RATISS-ONE
!pip install -q piper-tts
!python3 -m piper.download_voices en_US-lessac-medium --download-dir bouche/voix
!python3 tests_verdicts.py 2>&1 | tail -2

## 1. Le robinet à données : le tissu fabrique ses paires

In [ ]:
!python3 donnees/generer.py --verifier
!wc -l donnees/paires.jsonl

## 2. Éducation massive : des millions de relations en minutes
(Route B à l'échelle : le tissu grandit par phrases, délie 0 = régime école.)

In [ ]:
import time
t0 = time.time()
!python3 education/eduquer.py --rounds 10 2>&1 | grep -E "BILAN|TEMOINS"
print(f"éducation : {time.time() - t0:.1f} s")

## 3. Calibration des règles : le « fit » = 100 % de reproduction
Chaque paire (état → phrase) doit ressortir à l'identique, sinon ÉCHEC.

In [ ]:
import json
from bouche.regles import formuler
n = ok = 0
for ligne in open("donnees/paires.jsonl", encoding="utf-8"):
    p = json.loads(ligne)
    n += 1
    ok += formuler(p["etat"]) == p["phrase"]
print(f"règles : {ok}/{n} reproduites")
assert ok == n, "UNE RÈGLE A RATÉ"

## 4. Éval : batterie + 1 phrase parlée (preuve audio)

In [ ]:
!python3 tests_verdicts.py 2>&1 | tail -2
!python3 bouche/regles.py --demo EN --wav /tmp/preuve-colab.wav
from IPython.display import Audio
Audio("/tmp/preuve-colab.wav")